In [1]:
import math
import urllib.parse
import urllib.request
import json
import webbrowser
import os

In [2]:
# 1. Chuyển địa chỉ sang tọa độ
def dia_chi_sang_toa_do(dia_chi):
    print(f" Đang tìm tọa độ trên bản đồ cho: {dia_chi}...")
    # Thêm "Ho Chi Minh, Vietnam" để giới hạn tìm kiếm chuẩn xác hơn trong khu vực
    query = f"{dia_chi}, Ho Chi Minh, Vietnam"
    url = f"https://nominatim.openstreetmap.org/search?q={urllib.parse.quote(query)}&format=json&limit=1"
    
    req = urllib.request.Request(url, headers={'User-Agent': 'FastFood_App/1.0'})
    try:
        with urllib.request.urlopen(req) as response:
            data = json.loads(response.read().decode())
            if data:
                return (float(data[0]['lat']), float(data[0]['lon']))
    except Exception as e:
        print("Lỗi kết nối bản đồ:", e)
    return None

In [3]:
# 2. TÌM ĐƯỜNG ĐI NGẮN NHẤT THỰC TẾ
def tim_duong_ngan_nhat_osrm(toa_do_quan, toa_do_khach):
    print(" Đang tính toán lộ trình đường đi ngắn nhất...")
    url = f"http://router.project-osrm.org/route/v1/driving/{toa_do_quan[1]},{toa_do_quan[0]};{toa_do_khach[1]},{toa_do_khach[0]}?overview=full&geometries=geojson"
    
    req = urllib.request.Request(url, headers={'User-Agent': 'FastFood_App/1.0'})
    try:
        with urllib.request.urlopen(req) as response:
            data = json.loads(response.read().decode())
            if data['code'] == 'Ok':
                km_thuc_te = round(data['routes'][0]['distance'] / 1000, 2)
                phut_thuc_te = round(data['routes'][0]['duration'] / 60, 1)
                
                route_coords = data['routes'][0]['geometry']['coordinates']
                toa_do_ve_duong = [[lat, lon] for lon, lat in route_coords]
                
                return toa_do_ve_duong, km_thuc_te, phut_thuc_te
    except Exception as e:
        print("Lỗi tìm đường:", e)
    return None, None, None

def trang_thai_giao_hang(km):
    return "Đang giao" if km <= 10 else "Đã hủy"

In [ ]:
# ... (Các hàm dia_chi_sang_toa_do và tim_duong_ngan_nhat_osrm giữ nguyên) ...
def tao_va_mo_ban_do(toa_do_quan, toa_do_khach, dia_chi_khach, km, phut, mang_toa_do_duong_di):
    duong_di_json = json.dumps(mang_toa_do_duong_di)
    
    # Chuyển đổi phút sang giây để đếm ngược (giới hạn tối đa 10 phút theo yêu cầu của bạn)
    tong_giay = int(min(phut, 10) * 60) 

    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <title>Bản đồ Lộ Trình Giao Hàng</title>
        <meta charset="utf-8" />
        <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
        <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
        <style>
            body {{ font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; text-align: center; background: #f4f4f4; margin: 0; padding: 20px; }}
            #map {{ height: 65vh; width: 95%; margin: 0 auto; border: 2px solid #ccc; border-radius: 12px; box-shadow: 0 4px 15px rgba(0,0,0,0.1); }}
            
            .container {{ display: flex; flex-direction: column; align-items: center; gap: 15px; margin-bottom: 20px; }}
            
            .info-box {{ background: white; width: 95%; padding: 15px; border-radius: 12px; box-shadow: 0 2px 10px rgba(0,0,0,0.05); display: flex; justify-content: space-around; align-items: center; flex-wrap: wrap; }}
            
            .countdown-box {{ background: #fff5f5; border: 2px solid #e74c3c; padding: 10px 25px; border-radius: 50px; color: #e74c3c; }}
            .timer {{ font-size: 24px; font-weight: bold; }}
            
            h2 {{ margin: 0; color: #e74c3c; font-size: 20px; }}
            .detail-text {{ color: #555; margin: 5px 0; }}
        </style>
    </head>
    <body>
        <div class="container">
            <div class="info-box">
                <div style="text-align: left;">
                    <h2>🛵 Lộ trình giao hàng FastFood</h2>
                    <p class="detail-text">Đến: <b>{dia_chi_khach}</b></p>
                    <p class="detail-text">Khoảng cách: <b>{km} km</b></p>
                </div>
                
                <div class="countdown-box">
                    <div style="font-size: 12px; text-transform: uppercase; letter-spacing: 1px;">Thời gian còn lại</div>
                    <div id="countdown" class="timer">10:00</div>
                </div>
            </div>
        </div>

        <div id="map"></div>

        <script>
            // 1. Khởi tạo bản đồ
            var map = L.map('map').setView([{toa_do_quan[0]}, {toa_do_quan[1]}], 14);
            L.tileLayer('https://{{s}}.tile.openstreetmap.org/{{z}}/{{x}}/{{y}}.png').addTo(map);

            var shopIcon = L.icon({{ iconUrl: 'https://cdn-icons-png.flaticon.com/512/3448/3448650.png', iconSize: [40, 40] }});
            var homeIcon = L.icon({{ iconUrl: 'https://cdn-icons-png.flaticon.com/512/1177/1177577.png', iconSize: [40, 40] }});

            L.marker([{toa_do_quan[0]}, {toa_do_quan[1]}], {{icon: shopIcon}}).addTo(map).bindPopup("<b>Cửa hàng FastFood</b>");
            L.marker([{toa_do_khach[0]}, {toa_do_khach[1]}], {{icon: homeIcon}}).addTo(map).bindPopup("<b>Khách hàng</b>");

            var routeCoords = {duong_di_json};
            var polyline = L.polyline(routeCoords, {{ color: '#3498db', weight: 6, opacity: 0.8 }}).addTo(map);
            map.fitBounds(polyline.getBounds(), {{padding: [50, 50]}});

            // 2. Logic Đếm ngược (Countdown)
            let timeInSeconds = {tong_giay}; 
            const timerElement = document.getElementById('countdown');

            function updateTimer() {{
                const minutes = Math.floor(timeInSeconds / 60);
                const seconds = timeInSeconds % 60;

                // Định dạng hiển thị MM:SS
                timerElement.innerHTML = `${{minutes}}p ${{seconds < 10 ? '0' : ''}}${{seconds}}s`;
                if (timeInSeconds <= 0) {{
                    clearInterval(countdownInterval);
                    timerElement.innerHTML = "Đã đến nơi! 🍔";
                    timerElement.parentElement.style.backgroundColor = "#ebfbee";
                    timerElement.parentElement.style.borderColor = "#40c057";
                    timerElement.parentElement.style.color = "#40c057";
                }} else {{
                    timeInSeconds--;
                }}
            }}
            const countdownInterval = setInterval(updateTimer, 1000);
            updateTimer(); // Chạy ngay lập tức lần đầu
        </script>
    </body>
    </html>
    """
    file_name = "ban_do_giao_hang_realtime.html"
    with open(file_name, "w", encoding="utf-8") as f:
        f.write(html_content)
    
    print(f"\n✅ Đã tạo bản đồ với bộ đếm ngược!")
    webbrowser.open('file://' + os.path.realpath(file_name))

In [6]:
# CHƯƠNG TRÌNH CHÍNH
toa_do_quan = (10.7634, 106.6821)  # FastFood Universe (227 Nguyễn Văn Cừ, Quận 5)
print(" CHÀO MỪNG ĐẾN VỚI FASTFOOD UNIVERSE")
print(" Lưu ý: Cửa hàng chỉ giao trong phạm vi tối đa 10km.")
print("-" * 50)

# 1. NHẬP ĐỊA CHỈ TỰ DO
dia_chi_khach = input(" Nhập địa chỉ nhận hàng của bạn (VD: Chợ Bến Thành, Landmark 81, 123 Lê Lợi...): ").strip()
toa_do_khach = dia_chi_sang_toa_do(dia_chi_khach)

if not toa_do_khach:
    print(" Lỗi: Không tìm được địa chỉ trên bản đồ. Vui lòng thử nhập chi tiết hơn (gồm số nhà, tên đường, phường, quận).")
else:
    duong_di, km_thuc, phut_thuc = tim_duong_ngan_nhat_osrm(toa_do_quan, toa_do_khach)

    if not duong_di:
        print(" Lỗi: Không tìm được lộ trình giao thông đến địa chỉ này.")
    else:
        # 2. KIỂM TRA ĐIỀU KIỆN 10KM (Sử dụng khoảng cách thực tế)
        if km_thuc > 10:
            print(f"\n TỪ CHỐI ĐƠN HÀNG:")
            print(f"Khoảng cách đến chỗ bạn là {km_thuc} km. Rất tiếc, cửa hàng chỉ giao trong phạm vi 10 km đổ lại để đảm bảo chất lượng món ăn!")
        else:
            # 3. NẾU <= 10KM THÌ TIẾN HÀNH ĐẶT ĐƠN VÀ VẼ BẢN ĐỒ
            ket_qua = {
                "ten_quan": "FastFood Universe",
                "dia_chi_khach": dia_chi_khach,
                "khoang_cach_thuc_te_km": km_thuc,
                "thoi_gian_lai_xe_phut": phut_thuc,
                "trang_thai": trang_thai_giao_hang(km_thuc)
            }
            
            print("\n THÔNG TIN LỘ TRÌNH (ĐỦ ĐIỀU KIỆN GIAO):")
            print(json.dumps(ket_qua, ensure_ascii=False, indent=4))
            tao_va_mo_ban_do(toa_do_quan, toa_do_khach, dia_chi_khach, km_thuc, phut_thuc, duong_di)

 CHÀO MỪNG ĐẾN VỚI FASTFOOD UNIVERSE
 Lưu ý: Cửa hàng chỉ giao trong phạm vi tối đa 10km.
--------------------------------------------------
 Đang tìm tọa độ trên bản đồ cho: 980 Lac Long Quan...
 Đang tính toán lộ trình đường đi ngắn nhất...

 THÔNG TIN LỘ TRÌNH (ĐỦ ĐIỀU KIỆN GIAO):
{
    "ten_quan": "FastFood Universe",
    "dia_chi_khach": "980 Lac Long Quan",
    "khoang_cach_thuc_te_km": 5.23,
    "thoi_gian_lai_xe_phut": 6.2,
    "trang_thai": "Đang giao"
}

✅ Đã tạo bản đồ với bộ đếm ngược!
